# Finetune the model and compare every method

Run the cells in order, top to bottom. Nothing to install on your own computer.

Before you start: **Runtime -> Change runtime type -> T4 GPU**. It works without
a GPU too, just slower.


## 1. Get the code and the data

In [ ]:
!git clone --quiet https://github.com/annaymbern/Hackathon-Challenge1.git
%cd Hackathon-Challenge1
!git log --oneline -1

## 2. Install what the model needs

Takes a couple of minutes the first time. Ignore any warning about
restarting the runtime.


In [ ]:
!pip install --quiet sentence-transformers duckdb pandas

## 3. Check the data before training anything

This builds the train/test split and the training examples, and stops there.
If the last line does not say zero, do not go on: it would mean the model is
being tested on products it was trained on.


In [ ]:
!python src/finetune.py --dry-run

## 4. The methods that need no model

The numbers the model has to beat. Measured only on the held-out products,
so the comparison later is fair.


In [ ]:
!python src/benchmark.py --split test

## 5. Train the model on our products

A few minutes on a GPU, longer without one. It downloads the base model the
first time, which is about 500 MB.


In [ ]:
!python src/finetune.py --epochs 1

## 6. The comparison

Every method over the same held-out products. This is the table that decides
whether the finetuning was worth it.

The rule we fixed in Work 3: keep the finetuned model only if it beats the
baseline by 10 points or more. We also want it to beat Jaccard, which turned
out to be the stronger of the two simple methods.


In [ ]:
!python src/benchmark.py --methods all --split test --save results/benchmark_full.csv

## 7. See it working on one product

Sanity check: pick a yogurt and look at what each model suggests. Numbers
can hide nonsense, so look at the actual products.


In [ ]:
import sys
sys.path.insert(0, 'src')

from evaluation import load_products
from similarity import embed, rank_with

products = load_products()
query = 0  # change this number to try another yogurt

for label, model in [('pretrained', 'paraphrase-multilingual-MiniLM-L12-v2'),
                     ('finetuned', 'models/finetuned')]:
    vectors = embed(products['text'], model)
    print(f'--- {label} ---')
    print(f"query: {products['product_name'].iloc[query]}  "
          f"({products['brands'].iloc[query]})")
    for position, index in enumerate(rank_with(vectors, query, 3), start=1):
        row = products.iloc[index]
        score = float(vectors[index] @ vectors[query])
        print(f"  {position}. {row['product_name']}  ({row['brands']})  "
              f"similarity {score:.3f}  types: {'|'.join(sorted(row['types'])) or 'unknown'}")
    print()

## 8. Download the results

Saves the numbers to your computer so we can put them in the log and in the repo.


In [ ]:
from google.colab import files
files.download('results/benchmark_full.csv')
files.download('results/benchmark_full.summary.json')

---

**When it finishes:** copy the table from step 6 and paste it into the chat.
That is the number that decides what we build next.
